In [2]:
# =========================================================
# INSTALL LIBRARY
# =========================================================

!pip install -q \
streamlit \
langchain \
langchain-community \
langchain-google-genai \
chromadb \
pypdf \
google-generativeai \
pyngrok \
sentence-transformers

In [19]:
%%writefile app.py

import streamlit as st
import tempfile

from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

from langchain_google_genai import (
    GoogleGenerativeAIEmbeddings,
    ChatGoogleGenerativeAI
)

from langchain.chains import RetrievalQA

# =====================================================
# PAGE CONFIG
# =====================================================

st.set_page_config(
    page_title="Chatbot Panduan Tugas Akhir",
    page_icon="🎓",
    layout="wide"
)

# =====================================================
# HEADER
# =====================================================

st.title("🎓 Chatbot Panduan Tugas Akhir")
st.caption("Chatbot PDF dengan Gemini AI")

# =====================================================
# SESSION
# =====================================================

if "qa_chain" not in st.session_state:
    st.session_state.qa_chain = None

# =====================================================
# SIDEBAR
# =====================================================

with st.sidebar:

    st.header("⚙️ Pengaturan")

    api_key = st.text_input(
        "Gemini API Key",
        type="password"
    )

    uploaded_file = st.file_uploader(
        "Upload PDF",
        type=["pdf"]
    )

    process_btn = st.button("🚀 Bangun Chatbot")

# =====================================================
# PROCESS PDF
# =====================================================

if process_btn:

    if not api_key:
        st.error("Masukkan Gemini API Key")
        st.stop()

    if not uploaded_file:
        st.error("Upload PDF terlebih dahulu")
        st.stop()

    with st.spinner("Memproses PDF..."):

        # Simpan PDF sementara
        with tempfile.NamedTemporaryFile(
            delete=False,
            suffix=".pdf"
        ) as tmp:

            tmp.write(uploaded_file.read())
            pdf_path = tmp.name

        # Load PDF
        loader = PyPDFLoader(pdf_path)
        docs = loader.load()

        # Split text
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=200
        )

        splits = splitter.split_documents(docs)

        # Embedding Gemini
        embeddings = GoogleGenerativeAIEmbeddings(
             model="text-embedding-004",
             google_api_key=api_key
        )

        # Vector DB
        vectorstore = Chroma.from_documents(
            splits,
            embeddings
        )

        # Gemini LLM
        llm = ChatGoogleGenerativeAI(
            model="gemini-1.5-flash",
            google_api_key=api_key,
            temperature=0.3
        )

        # QA Chain
        qa_chain = RetrievalQA.from_chain_type(
            llm=llm,
            retriever=vectorstore.as_retriever()
        )

        st.session_state.qa_chain = qa_chain

        st.success("✅ Chatbot berhasil dibuat!")

# =====================================================
# CHAT AREA
# =====================================================

if st.session_state.qa_chain:

    user_question = st.chat_input(
        "Tanyakan sesuatu tentang PDF..."
    )

    if user_question:

        with st.spinner("AI sedang berpikir..."):

            answer = st.session_state.qa_chain.run(
                user_question
            )

        st.chat_message("user").write(user_question)

        st.chat_message("assistant").write(answer)

Overwriting app.py


In [18]:
from pyngrok import ngrok
import subprocess
import threading
import time

# ==========================================
# MASUKKAN TOKEN NGROK BARU
# ==========================================

NGROK_AUTHTOKEN = "3Dcy9cdYBBOUpozR2DbCnsD8l3M_84HddUnZfruv54rmp7K3M"

ngrok.set_auth_token(NGROK_AUTHTOKEN)

# ==========================================
# HAPUS TUNNEL LAMA
# ==========================================

ngrok.kill()

# ==========================================
# RUN STREAMLIT
# ==========================================

def run_streamlit():
    subprocess.Popen([
        "streamlit",
        "run",
        "app.py",
        "--server.port",
        "8501"
    ])

threading.Thread(target=run_streamlit).start()

time.sleep(10)

# ==========================================
# BUAT LINK PUBLIK
# ==========================================

public_url = ngrok.connect(8501)

print("🚀 Chatbot berhasil berjalan!")
print(public_url)

🚀 Chatbot berhasil berjalan!
NgrokTunnel: "https://headpiece-blurt-enduring.ngrok-free.dev" -> "http://localhost:8501"


In [9]:
!grep -n "embedding-001" app.py

In [10]:
!pkill streamlit

print("✅ Streamlit lama dihentikan")

✅ Streamlit lama dihentikan


In [11]:
!pkill ngrok

print("✅ Ngrok lama dihentikan")

✅ Ngrok lama dihentikan


In [12]:
from pyngrok import ngrok

ngrok.kill()

print("✅ Semua tunnel dibersihkan")

✅ Semua tunnel dibersihkan


In [14]:
with open("app.py", "r") as f:
    code = f.read()

code = code.replace(
    'model="text-embedding-004"',
    'model="models/text-embedding-004"'
)

with open("app.py", "w") as f:
    f.write(code)

print("✅ Embedding model FINAL berhasil diperbaiki")

✅ Embedding model FINAL berhasil diperbaiki


In [15]:
!grep -n "embedding" app.py

98:        embeddings = GoogleGenerativeAIEmbeddings(
99:             model="models/text-embedding-004",
106:            embeddings


In [16]:
!pkill streamlit
!pkill ngrok

In [17]:
from pyngrok import ngrok
ngrok.kill()